In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/Intent-Classification-ML-Project/

/content/drive/MyDrive/Intent-Classification-ML-Project


In [3]:
!ls


data  notebooks  README.md  requirements.txt  src


# Loading Stored Dataset with v1 features

In [4]:
import pandas as pd

# Paths should match what you used in 01
features_path = "./data/rba_features_v1.parquet"

df = pd.read_parquet(features_path)

print("Loaded shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())  # first 40 cols, just to confirm

df.head()

Loaded shape: (300000, 57)

Columns:
['Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'browser', 'os', 'hour', 'dayofweek', 'is_new_device_for_user', 'is_new_ip_for_user', 'is_off_hours', 'failed_login', 'failures_last_5', 'failure_streak', 'failure_streak_capped', 'location', 'new_location_flag', 'new_asn_flag', 'device_fingerprint', 'valid_device', 'new_device_flag', 'device_change_rate', 'ts_sec', 'delta_sec', 'new_window', 'window_id', 'logins_5min', 'failure_flag', 'burst_failure_count', 'failure_rate', 'streak_reset', 'streak_id', 'failure_streak_length', 'location_freq', 'location_rarity', 'device_type_freq', 'device_type_rarity', 'asn_freq', 'asn_rarity', 'location_rarity_q', 'device_type_rarity_q', 'asn_rarity_q', 'user_offhour_rate', 'is_unusual_time_for_user', 'user_hour_std',

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,device_type_rarity,asn_freq,asn_rarity,location_rarity_q,device_type_rarity_q,asn_rarity_q,user_offhour_rate,is_unusual_time_for_user,user_hour_std,user_hour_std_q
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0.000013,15067,0.000066,1,0,0,0.0,0,0.0,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0.000005,36,0.027778,0,0,2,0.0,0,0.5,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0.000005,36,0.027778,0,0,2,0.0,0,0.5,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0.000005,86455,0.000012,3,0,0,0.0,0,0.0,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0.000005,86455,0.000012,0,0,0,0.0,0,0.0,0


# Define the binary rule indicators (Iᵢ)



> rule_unusual_time – off-hours & unusual for this user
	2.	rule_off_hours – off-hours, but not unusual (user sometimes uses nights)
	3.	rule_new_device – device changed since last login
	4.	rule_new_asn – ASN changed since last login
	5.	rule_recent_failures – at least 3 failures in last 5 logins



In [5]:
# --- 1) Rule: Unusual time for this user ---
# 1 if this login is off-hours AND user rarely uses off-hours (< 0.2),
# we already encoded this as is_unusual_time_for_user
df["rule_unusual_time"] = df["is_unusual_time_for_user"].astype(int)

# --- 2) Rule: Off-hours (but NOT already counted as unusual) ---
# Example: user sometimes logs at night, so it's off-hours but not rare for them
df["rule_off_hours"] = (
    (df["is_off_hours"] == 1) &
    (df["is_unusual_time_for_user"] == 0)
).astype(int)

# --- 3) Rule: New device since last login ---
df["rule_new_device"] = df["new_device_flag"].astype(int)

# --- 4) Rule: New ASN since last login ---
df["rule_new_asn"] = df["new_asn_flag"].astype(int)

# --- 5) Rule: Recent failures: 3 or more in previous 5 logins ---
# Make sure NaNs in failures_last_5 are treated as 0
failures_last_5_clean = df["failures_last_5"].fillna(0)
df["rule_recent_failures"] = (failures_last_5_clean >= 3).astype(int)

# Quick sanity check: show first few columns
df[[
    "is_off_hours",
    "is_unusual_time_for_user",
    "failures_last_5",
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures"
]].head()

,is_off_hours,is_unusual_time_for_user,failures_last_5,rule_unusual_time,rule_off_hours,rule_new_device,rule_new_asn,rule_recent_failures
0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0


Validating the Distributions just to make sure not all are 0's or 1's. We want evenly distributed!

In [6]:
rule_cols = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
]

for col in rule_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


rule_unusual_time value counts:
rule_unusual_time
0    282476
1     17524
Name: count, dtype: int64

rule_off_hours value counts:
rule_off_hours
0    278226
1     21774
Name: count, dtype: int64

rule_new_device value counts:
rule_new_device
0    184842
1    115158
Name: count, dtype: int64

rule_new_asn value counts:
rule_new_asn
0    201323
1     98677
Name: count, dtype: int64

rule_recent_failures value counts:
rule_recent_failures
0    189779
1    110221
Name: count, dtype: int64


Compute rule_risk_score


In [7]:
# Compute rule-based risk score as weighted sum of rule indicators

df["rule_risk_score"] = (
    2 * df["rule_unusual_time"]
    + 1 * df["rule_off_hours"]
    + 1 * df["rule_new_device"]
    + 1 * df["rule_new_asn"]
    + 1 * df["rule_recent_failures"]
)

print("rule_risk_score summary:")
print(df["rule_risk_score"].describe())

print("\nCounts per rule_risk_score:")
print(df["rule_risk_score"].value_counts().sort_index())

rule_risk_score summary:
count    300000.000000
mean          1.269593
std           1.537271
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max           5.000000
Name: rule_risk_score, dtype: float64

Counts per rule_risk_score:
rule_risk_score
0    154037
1     37065
2     13555
3     78883
4      2246
5     14214
Name: count, dtype: int64


	•	Median = 0 → for at least 50% of logins, none of the 5 rules fired.
	•	These are “totally clean” from the baseline’s point of view.
	•	75th percentile = 3 → the top 25% of logins have score ≥ 3, meaning:
	•	multiple rules triggered together (e.g., off-hours + new device + failures).
	•	Mean ~1.27, std ~1.54 → most logins are low-to-moderate risk; a smaller group has piled-up risk.

This is good:
We don’t want everything to look risky, but we do want a clear separation between “no rule triggered” and “multiple rules triggered”.

🔹 Score = 0 → 154,037 logins (~51.3%)
	•	None of the rules fired:
	•	Not off-hours
	•	Not unusual time
	•	No new device
	•	No new ASN
	•	No heavy recent failures

👉 These are your “baseline-normal” logins.
This is exactly what we want: about half the data looks totally unremarkable.

⸻

🔹 Score = 1 → 37,065 logins (~12.4%)
	•	Exactly one mild rule fired:
	•	Example: just off-hours but not unusual for that user,
	•	or just new device,
	•	or just new ASN,
	•	or just 3+ recent failures.

👉 These are slightly risky but not alarming.
Good place to label as low risk but “watch”.

⸻

🔹 Score = 2 → 13,555 logins (~4.5%)
	•	Either:
	•	Unusual time only (2 points), or
	•	two mild rules (e.g., new device + new ASN).

👉 This is your first “this looks genuinely suspicious” layer.

Remember: unusual time on its own got +2 because in EDA it had ~2× attack rate.

⸻

🔹 Score = 3 → 78,883 logins (~26.3%)
	•	Examples:
	•	unusual time (2) + new device (1)
	•	unusual time (2) + recent failures (1)
	•	or 3 different +1 rules together (off-hours + new ASN + recent failures, etc.)

👉 This is a big chunk of the dataset (~1/4) where multiple risk factors align.

This is the region where “step-up auth” is very natural: user might still be legit, but we want an extra check.

⸻

🔹 Score = 4 → 2,246 logins (~0.75%)
	•	Example:
	•	unusual time (2) + any two of {off-hours, new device, new ASN, recent failures}
	•	or four mild rules firing at the same time.

👉 Very clustered risk; these are strong candidates for:
	•	Step-up auth at minimum, often block / review.

⸻

🔹 Score = 5 → 14,214 logins (~4.7%)
	•	This is near-max risk:
	•	E.g., unusual time (2) + three other risk factors (off-hours + new device + recent failures), etc.

👉 These are your top-risk logins.
In a real system, many of these would probably be outright blocked or heavily challenged.

# Potential extra rule signals (for v2 baseline later)
	1.	Device type risk
	•	From EDA: Device Type = bot and unknown had very high attack rates.
	•	Rule idea:
	•	rule_risky_device_type = 1 if device_type in {bot, unknown}
	2.	Global rarity
	•	asn_rarity or location_rarity:
	•	Very rare ASNs or locations might be suspicious.
	•	Rule idea:
	•	rule_rare_asn = 1 if asn_rarity > some_threshold
	•	rule_rare_location = 1 if location_rarity > some_threshold
	3.	Stronger failure pattern
	•	You already have failure_streak_capped and burst_failure_count.
	•	Rule idea:
	•	rule_long_streak = 1 if failure_streak_capped == 5
	•	rule_burst_failures = 1 if burst_failure_count >= X
	4.	Country-based risk
	•	Some countries in EDA (e.g., small set with 50%+ attack rate) were much riskier.
	•	Rule idea:
	•	rule_high_risk_country = 1 if Country in {list_of_very_high_attack_rate_countries}

# Mapping rule_risk_score → rule_risk_band (low / medium / high)

In [8]:
def map_rule_band(score: int) -> str:
    if score <= 1:
        return "low"
    elif score <= 3:
        return "medium"
    else:
        return "high"

df["rule_risk_band"] = df["rule_risk_score"].apply(map_rule_band)

print("Risk band counts:")
print(df["rule_risk_band"].value_counts())

Risk band counts:
rule_risk_band
low       191102
medium     92438
high       16460
Name: count, dtype: int64


So we found
	•	Low risk → ~63.7% of logins
	•	Medium risk → ~30.8%
	•	High risk → ~5.5%

# Checking Check attack rate per band using Is Attack IP

In [9]:
label = "Is Attack IP"

print("\nAttack rate by rule_risk_band:")
print(df.groupby("rule_risk_band")[label].mean())

print("\nCrosstab of rule_risk_band vs label:")
print(pd.crosstab(df["rule_risk_band"], df[label]))


Attack rate by rule_risk_band:
rule_risk_band
high      0.183111
low       0.075337
medium    0.110788
Name: Is Attack IP, dtype: float64

Crosstab of rule_risk_band vs label:
Is Attack IP     False  True 
rule_risk_band               
high             13446   3014
low             176705  14397
medium           82197  10241


Overall attack rate in the dataset ≈ 9.2%.

So compared to the overall 9.2%:
	•	Low risk band (~7.5%)
	•	Slightly below global rate.
	•	This means the rules successfully push some attacks out of the low band, but not dramatically.
	•	Interpretation: low band is “mostly normal, but not clean enough to blindly trust”.
	•	Medium risk band (~11.1%)
	•	Above global rate.
	•	This is your main “suspicious but not crazy” zone – fits naturally with “step-up authentication”.
	•	High risk band (~18.3%)
	•	About 2× the global average.
	•	This is where the rules clearly concentrate higher risk logins.
	•	Great candidate for “block or heavily challenge”.

So the baseline does separate risk in the right direction:

low < overall < medium < high

That’s already a good sign.

	•	Risk is monotonic:
low → medium → high have increasing attack rates.
	•	High band is clearly high risk (~18.3% vs global 9.2%).
	•	Band distribution (63% low, 31% medium, 5.5% high) is realistic for:
	•	Allow most logins,
	•	Step-up a subset,
	•	Hard-check a small tail.

What’s not perfect (and expected at this stage) 🤏
	•	Low band still has 7.5% attack rate, not dramatically below global 9.2%.
	•	Means our rules do some filtering, but not super aggressive.
	•	Only 11% of all attacks fall into “high”.
	•	For a stronger baseline, you might want a higher fraction of attacks in medium+high bands.


# “We implemented a rule-based baseline v1 that uses five interpretable signals: unusual login time, off-hours access, new device, new ASN, and recent failure bursts. Each rule contributes a fixed number of points to a rule risk score, which is then mapped into low, medium, and high risk bands.

In our experiments on 300,000 login events, the overall attack rate is about 9.2%. In the low, medium, and high risk bands, the attack rates are approximately 7.5%, 11.1%, and 18.3% respectively. This shows that the rule-based baseline successfully concentrates higher-risk logins in the medium and high bands, but also that a significant portion of attacks still fall into the low band. This motivates the need for a learned intent classifier that can exploit richer temporal and contextual patterns than fixed rules.”